# Fine-Tune a Small LLM for Extraction, with LoRA

In the extraction lab, a 4-billion-parameter LLM extracted the entities of the 1914 press. It also labelled 2,000 other passages: these labels will teach the task to a model **six times smaller**, Qwen3-0.6B, with LoRA. That's distillation: the large model serves once, the small one serves in production.

1. The small model as is: how good is it?
2. Prepare the training data
3. LoRA: how many parameters do we train?
4. Train
5. The fine-tuned small model against the large one

This notebook needs a GPU: *Runtime > Change runtime type > T4 GPU*.

## Data and Model

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/gold.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/distill.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/teacher-gold.jsonl
%pip install -q peft trl rapidfuzz
%pip uninstall -q -y torchao  # Colab's version is too old for peft

In [ ]:
import json
import time

import pandas as pd
import torch
from rapidfuzz import fuzz
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")

gold = pd.read_json("gold.jsonl", lines=True)
distill = pd.read_json("distill.jsonl", lines=True)
teacher = pd.read_json("teacher-gold.jsonl", lines=True)
FIELDS = ["people", "places", "organizations"]
print(len(gold), "test passages,", len(distill), "passages labelled by the large model")

The extraction lab's measure (F1 per field, lenient matching), and the prompt (the one given to the large model, translated).

In [ ]:
def normalize(name: str) -> str:
    return " ".join(name.lower().replace("’", "'").split())


def same(a: str, b: str) -> bool:
    a, b = normalize(a), normalize(b)
    return fuzz.ratio(a, b) >= 85 or (min(len(a), len(b)) >= 4 and (a in b or b in a))


def evaluate(predictions: list[dict[str, list[str]]]) -> pd.Series:
    scores = {}
    for field in FIELDS:
        found = expected = right_found = right_expected = 0
        for prediction, (_, row) in zip(predictions, gold.iterrows()):
            value = prediction.get(field)  # a small model can answer anything
            predicted = list(dict.fromkeys(p for p in value if isinstance(p, str))) if isinstance(value, list) else []
            truth = row[field]
            found += len(predicted)
            expected += len(truth)
            right_found += sum(any(same(p, t) for t in truth) for p in predicted)
            right_expected += sum(any(same(p, t) for p in predicted) for t in truth)
        precision, recall = right_found / max(found, 1), right_expected / max(expected, 1)
        scores[field] = 2 * precision * recall / max(precision + recall, 1e-9)
    return pd.Series(scores).round(2)


INSTRUCTIONS = """Extract the named entities of this passage of the 1914 French press.
- people: persons, without their title (M., Mme, sir, lord);
- places: places (countries, regions, towns, streets, buildings);
- organizations: named organizations (assemblies, parties, unions, newspapers, agencies).
Each entity once, as written in the text, errors included.
No generic mentions such as "le gouvernement". Do not invent anything.
Answer only with a JSON object with the keys people, places and organizations."""


def messages(text: str) -> list[dict[str, str]]:
    return [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": text}]


print("Large model:", dict(evaluate(teacher.to_dict("records"))))

## 1. The Small Model as Is

Write `extract_all(texts)` that generates, in batches of 8, the model's answer to each passage (`tokenizer.apply_chat_template(..., add_generation_prompt=True, enable_thinking=False)`, then `model.generate(..., max_new_tokens=256, do_sample=False)`), and reads it as JSON (`json.loads`, an empty dictionary if the answer isn't valid JSON). Measure the F1 on the 40 test passages, the share of invalid answers and the time per passage.

In [ ]:
# Your code here

### Solution

In [ ]:
def extract_all(texts: list[str], batch_size: int = 8) -> tuple[list[dict], int]:
    results, invalid = [], 0
    for start in range(0, len(texts), batch_size):
        prompts = [
            tokenizer.apply_chat_template(messages(t), tokenize=False, add_generation_prompt=True, enable_thinking=False)
            for t in texts[start : start + batch_size]
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        for output in outputs[:, inputs.input_ids.shape[1] :]:
            answer = tokenizer.decode(output, skip_special_tokens=True).strip()
            answer = answer.removeprefix("```json").removesuffix("```").strip()
            try:
                parsed = json.loads(answer)
                results.append(parsed if isinstance(parsed, dict) else {})
            except json.JSONDecodeError:
                results.append({})
                invalid += 1
    return results, invalid


start = time.time()
base_predictions, invalid = extract_all(gold.text.tolist())
base_seconds = (time.time() - start) / len(gold)
print(f"{invalid} invalid answers, {base_seconds:.2f} s per passage")
evaluate(base_predictions)

## 2. Prepare the Training Data

Each training example is a conversation: the prompt (instructions and passage), and the expected answer, the large model's JSON object. Build a `datasets.Dataset` with the columns `prompt` (the list of messages) and `completion` (a list of one `assistant` message, whose content is `json.dumps(labels, ensure_ascii=False)`). Keep 100 examples for validation.

In [ ]:
# Your code here

### Solution

In [ ]:
from datasets import Dataset


def example(row) -> dict:
    labels = {field: row[field] for field in FIELDS}
    return {
        "prompt": messages(row["text"]),
        "completion": [{"role": "assistant", "content": json.dumps(labels, ensure_ascii=False)}],
    }


dataset = Dataset.from_list([example(row) for row in distill.to_dict("records")])
split = dataset.train_test_split(test_size=100, seed=0)
print(split)
print(split["train"][0]["completion"][0]["content"])

## 3. LoRA

With `peft`, add rank-16 LoRA adapters to the model's attention projections (`LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")`). How many parameters are trained, out of how many?

In [ ]:
# Your code here

### Solution

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

## 4. Train

Train one epoch with `trl.SFTTrainer`: `SFTConfig(num_train_epochs=1, per_device_train_batch_size=8, gradient_accumulation_steps=2, learning_rate=2e-4, fp16=True, logging_steps=10, eval_strategy="steps", eval_steps=40, max_length=1024, report_to="none", output_dir="lora")`. The loss is computed on the answer (`completion`) only, not on the prompt. Plot the training and validation loss.

In [ ]:
# Your code here

### Solution

In [ ]:
import matplotlib.pyplot as plt
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="lora",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=40,
    max_length=1024,
    report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=split["train"], eval_dataset=split["test"], processing_class=tokenizer)
start = time.time()
trainer.train()
print(f"{(time.time() - start) / 60:.1f} min of training")

history = pd.DataFrame(trainer.state.log_history)
plt.plot(history.dropna(subset=["loss"]).step, history.dropna(subset=["loss"]).loss, label="training")
plt.plot(history.dropna(subset=["eval_loss"]).step, history.dropna(subset=["eval_loss"]).eval_loss, "o-", label="validation")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

## 5. The Fine-Tuned Small Model against the Large One

Rerun the extraction of the 40 test passages with the fine-tuned model. Compare, in one table, the small model's F1 before and after fine-tuning and the large model's, with the time per passage. Did fine-tuning close the gap? What do we gain by putting the small model in production?

In [ ]:
# Your code here

### Solution

In [ ]:
# The adapters merged into the weights: no separate LoRA computation at each layer.
model = model.merge_and_unload()
model.eval()
start = time.time()
tuned_predictions, invalid = extract_all(gold.text.tolist())
tuned_seconds = (time.time() - start) / len(gold)
print(f"{invalid} invalid answers, {tuned_seconds:.2f} s per passage")

pd.DataFrame({
    "Qwen3-0.6B": evaluate(base_predictions),
    "Qwen3-0.6B + LoRA": evaluate(tuned_predictions),
    "large model (4B)": evaluate(teacher.to_dict("records")),
})

# Fine-tuning closes most of the gap with the large model, on this narrow task, for a
# model six times smaller: in production, it fits a much cheaper machine, or handles more
# passages at once on the same GPU. The large model's labels aren't perfect (a person
# filed under organizations...): the student learns its mistakes too.